# Hướng dẫn đăng ký Strategy & Lập Profile

Tutorial này bao gồm:

1. **Registry** – đăng ký & tra cứu strategy trong hệ thống
2. **Tạo strategy tùy chỉnh** – viết technique mới kế thừa `BaseTechnique`
3. **Profiler** – chạy profiling hàng loạt, tính SignalProfile
4. **Lưu profile** – xuất JSON + PDF report đầy đủ
5. **Đọc & so sánh profile** – load lại từ local

> **Yêu cầu**: Đã có dữ liệu OHLCV trong `data/ohlcv/` (xem `data_tutorial.ipynb`).

## 1. Xem danh sách strategy đã đăng ký

Mọi strategy kế thừa `BaseTechnique` và dùng decorator `@register("tên")` sẽ tự động được thêm vào global registry.

Các strategy built-in: `sma_crossover`, `rsi_crossover`, `macd_crossover`.

In [1]:
from vnstock_forecast.forecast.technical.registry import (
    get_all_techniques,
    get_technique,
    list_technique_names,
)

# Import strategies để chúng tự đăng ký vào registry
from vnstock_forecast.forecast.technical.strategies import (
    SMACrossover, RSICrossover, MACDCrossover,
)

# Xem tất cả technique đã đăng ký
print("Techniques đã đăng ký:", list_technique_names())

# Xem chi tiết từng technique
for name, cls in get_all_techniques().items():
    instance = cls()
    print(f"\n  {name}:")
    print(f"    Class:    {cls.__name__}")
    print(f"    Params:   {instance.params}")
    print(f"    Lookback: {instance.required_lookback}")

Techniques đã đăng ký: ['macd_crossover', 'rsi_crossover', 'sma_crossover', 'sma_short_crossover']

  macd_crossover:
    Class:    MACDCrossover
    Params:   {'fast_period': 12, 'slow_period': 26, 'signal_period': 9, 'sl_pct': 0.07, 'tp_pct': 0.1}
    Lookback: 40

  rsi_crossover:
    Class:    RSICrossover
    Params:   {'period': 14, 'oversold': 30.0, 'overbought': 70.0, 'sl_pct': 0.07, 'tp_pct': 0.1}
    Lookback: 19

  sma_crossover:
    Class:    SMACrossover
    Params:   {'period': 20, 'sl_pct': 0.07, 'tp_pct': 0.1}
    Lookback: 22

  sma_short_crossover:
    Class:    SMAShortCrossover
    Params:   {'short_period': 10, 'long_period': 30, 'sr_lookback': 60, 'vol_lookback': 20, 'surge_threshold': 1.5, 'max_strength': 0.03, 'min_confidence': 0.5}
    Lookback: 60


## 2. Tạo strategy tùy chỉnh

Để tạo strategy mới, cần:
1. Kế thừa `BaseTechnique`
2. Dùng `@register("tên")` để đăng ký
3. Implement `analyze_step(ctx, symbol)` → trả về `list[Signal]`

In [2]:
# from vnstock_forecast.forecast.technical.base import BaseTechnique
# from vnstock_forecast.forecast.technical.registry import register
# from vnstock_forecast.forecast.signal import Signal, SignalDirection, TradePlan
# from vnstock_forecast.engine.backtest.context import StepContext


# @register("double_sma_crossover")
# class DoubleSMACrossover(BaseTechnique):
#     """
#     Double SMA Crossover – tín hiệu khi SMA nhanh cắt SMA chậm.

#     - BUY:  SMA(fast) cắt lên trên SMA(slow)
#     - SELL: SMA(fast) cắt xuống dưới SMA(slow)
#     """

#     name = "double_sma_crossover"

#     def __init__(self, fast: int = 10, slow: int = 30, sl_pct: float = 0.07, tp_pct: float = 0.12):
#         self.fast = fast
#         self.slow = slow
#         self.sl_pct = sl_pct
#         self.tp_pct = tp_pct
#         self.required_lookback = slow + 2

#     @property
#     def params(self):
#         return {"fast": self.fast, "slow": self.slow, "sl_pct": self.sl_pct, "tp_pct": self.tp_pct}

#     def analyze_step(self, ctx: StepContext, symbol: str) -> list[Signal]:
#         df = ctx.history(symbol, lookback=self.slow + 5)
#         if len(df) < self.slow + 1:
#             return []v", res

#         closes = df["Close"]
#         sma_fast = closes.rolling(self.fast).mean()
#         sma_slow = closes.rolling(self.slow).mean()

#         if sma_fast.isna().iloc[-1] or sma_slow.isna().iloc[-1]:
#             return []

#         prev_fast, curr_fast = sma_fast.iloc[-2], sma_fast.iloc[-1]
#         prev_slow, curr_slow = sma_slow.iloc[-2], sma_slow.iloc[-1]
#         price = ctx.price(symbol)
#         signals = []

#         # Golden cross: SMA fast cắt lên trên SMA slow
#         if prev_fast <= prev_slow and curr_fast > curr_slow:
#             signals.append(Signal(
#                 technique=self.name,
#                 symbol=symbol,
#                 direction=SignalDirection.BUY,
#                 timestamp=ctx.timestamp,
#                 trade_plan=TradePlan(
#                     entry=price,
#                     stop_loss=round(price * (1 - self.sl_pct), 2),
#                     take_profit=round(price * (1 + self.tp_pct), 2),
#                 ),
#                 confidence=0.6,
#                 reason=f"Golden cross: SMA({self.fast}) > SMA({self.slow})",
#             ))

#         # Death cross: SMA fast cắt xuống dưới SMA slow
#         if prev_fast >= prev_slow and curr_fast < curr_slow:
#             signals.append(Signal(
#                 technique=self.name,
#                 symbol=symbol,
#                 direction=SignalDirection.SELL,
#                 timestamp=ctx.timestamp,
#                 confidence=0.5,
#                 reason=f"Death cross: SMA({self.fast}) < SMA({self.slow})",
#             ))

#         return signals


# # Kiểm tra đăng ký thành công
# print("Registry sau khi thêm:", list_technique_names())

## 3. Chuẩn bị dữ liệu OHLCV

Query dữ liệu daily cho 1-2 mã cổ phiếu để chạy profiling.

In [3]:
import pandas as pd
from vnstock_forecast.engine.data.query import query_ohlcv_grouped
from vnstock_forecast.config import load_config, to_app_config
cfg = to_app_config(load_config())
# Query dữ liệu từ 2025-01-01 đến nay
from_ts = int(pd.Timestamp("2024-01-01").timestamp())
grouped = query_ohlcv_grouped(
    symbols=cfg.data.updater.symbols,
    resolutions="D",
    from_ts=from_ts,
)
data = grouped["D"]  # dict: {"VHM": DataFrame, "SHB": DataFrame}

for sym, df in data.items():
    print(f"{sym}: {df.shape[0]} bars | {df.index[0]} → {df.index[-1]}")



ACB: 541 bars | 1704153600 → 1773187200
BID: 541 bars | 1704153600 → 1773187200
CTG: 541 bars | 1704153600 → 1773187200
DGC: 541 bars | 1704153600 → 1773187200
FPT: 541 bars | 1704153600 → 1773187200
GAS: 541 bars | 1704153600 → 1773187200
GVR: 541 bars | 1704153600 → 1773187200
HDB: 541 bars | 1704153600 → 1773187200
HPG: 541 bars | 1704153600 → 1773187200
LPB: 541 bars | 1704153600 → 1773187200
MBB: 541 bars | 1704153600 → 1773187200
MSN: 541 bars | 1704153600 → 1773187200
MWG: 541 bars | 1704153600 → 1773187200
PLX: 541 bars | 1704153600 → 1773187200
SAB: 541 bars | 1704153600 → 1773187200
SHB: 540 bars | 1704153600 → 1773187200
SSB: 541 bars | 1704153600 → 1773187200
SSI: 541 bars | 1704153600 → 1773187200
STB: 541 bars | 1704153600 → 1773187200
TCB: 541 bars | 1704153600 → 1773187200
TPB: 541 bars | 1704153600 → 1773187200
VCB: 541 bars | 1704153600 → 1773187200
VHM: 540 bars | 1704153600 → 1773187200
VIB: 541 bars | 1704153600 → 1773187200
VIC: 541 bars | 1704153600 → 1773187200


## 4. Chạy Profiler – Profile tất cả strategy

`Profiler.run()` sẽ:
- Lặp qua mỗi technique đã đăng ký (hoặc danh sách chỉ định)
- Tạo `AnalysisBot` wrapper cho từng technique → chạy backtest
- Tính `SignalProfile` (win rate, avg return, R:R, frequency...)
- Lưu kết quả vào `profiler.profiles` (JSON) và `profiler.results` (đầy đủ cho PDF)

In [4]:
from vnstock_forecast.forecast.profiler import Profiler
from vnstock_forecast.forecast.technical.strategies import SMAShortCrossover
profiler = Profiler(
    initial_cash=100_000_000,
    commission_rate=0.0015,
    settlement_days=3,
)

# Chạy profiling cho TẤT CẢ technique trong registry
profiles = profiler.run(
    data=data,
    start="2024-04-01",
    end="2026-03-01",
    techniques=[SMAShortCrossover(min_confidence=0.65)]
)

### 4.1 Xem kết quả profile

In [5]:
# In summary từng technique
for name, profile in profiles.items():
    profile.print_summary()
    print()

  SIGNAL PROFILE: sma_short_crossover
  Params:         {'short_period': 10, 'long_period': 30, 'sr_lookback': 60, 'vol_lookback': 20, 'surge_threshold': 1.5, 'max_strength': 0.03, 'min_confidence': 0.65}
  Symbols:        ACB, BID, CTG, DGC, FPT, GAS, GVR, HDB, HPG, LPB, MBB, MSN, MWG, PLX, SAB, SHB, SSB, SSI, STB, TCB, TPB, VCB, VHM, VIB, VIC, VJC, VNM, VPB, VPL, VRE
  Period:         2024-04-01 → 2026-02-27
  Total Bars:     474
  Created:        2026-03-12T15:44:40.269380
------------------------------------------------------------
  [BUY]
    Signals:            50
    Wins:               20
    Losses:             27
    Win Rate:       42.5%
    Avg Return:      2.08%
    Avg Win:        10.00%
    Avg Loss:       -3.79%
    R:R Ratio:        2.64
    Frequency:     0.1055
  [SELL]
    Signals:             0
    Wins:                0
    Losses:              0
    Win Rate:        0.0%
    Avg Return:      0.00%
    Avg Win:         0.00%
    Avg Loss:        0.00%
    R:R Rati

### 5.2 Tạo PDF cho 1 technique cụ thể

### 5.3 Tạo PDF với benchmark (tính Alpha & Beta)

Truyền thêm `benchmark_data` (VNINDEX daily) để PDF tính alpha/beta.

In [6]:
bm_grouped = query_ohlcv_grouped(symbols="VNINDEX", resolutions="D", from_ts=from_ts)
benchmark = bm_grouped["D"]["VNINDEX"]
benchmark

,Open,High,Low,Close,Volume
Timestamp,,,,,
1704153600,1138.01,1139.71,1128.69,1131.72,7.773372e+08
1704240000,1131.97,1144.17,1128.32,1144.17,6.575681e+08
1704326400,1145.49,1160.08,1144.17,1150.72,1.141609e+09
1704412800,1151.43,1155.84,1149.08,1154.68,7.602826e+08
1704672000,1157.54,1162.56,1155.48,1160.19,8.144106e+08
...,...,...,...,...,...
1772668800,1837.34,1849.01,1808.07,1808.51,1.097225e+09
1772755200,1801.08,1807.31,1767.84,1767.84,9.527709e+08
1773014400,1757.73,1767.84,1651.68,1652.79,1.278598e+09


In [7]:
# Query VNINDEX làm benchmark
if "D" in bm_grouped and "VNINDEX" in bm_grouped["D"]:
    benchmark = bm_grouped["D"]["VNINDEX"]
    profiler.save(benchmark_data=benchmark)
    print("Đã lưu profile + PDF với alpha/beta vs VNINDEX")
else:
    print("Không tìm thấy dữ liệu VNINDEX, bỏ qua benchmark")
    profiler.save()

Bắt đầu tạo PDF report cho technique '%s'... sma_short_crossover
Tạo trang tổng quan (overview)...
Tạo trang metrics...
Tạo trang biểu đồ tín hiệu (signal charts)...
Tạo trang lịch sử giao dịch (trade history)...
Tạo trang nhật ký sự kiện (event log)...
Tạo trang danh sách tín hiệu (signal list)...
Đã lưu profile + PDF với alpha/beta vs VNINDEX


## 6. Đọc lại profile từ local

Profile JSON có thể load lại bất cứ lúc nào mà không cần chạy lại backtest.

In [15]:
from vnstock_forecast.forecast.profile import SignalProfile

# Load tất cả profile trong thư mục
loaded = SignalProfile.load_all(profiler.profile_dir)
print(f"Đã load {len(loaded)} profiles:")
for name, p in loaded.items():
    print(f"  {name}: win_rate={p.overall_win_rate:.1%}, period={p.period}")

Đã load 4 profiles:
  double_sma_crossover: win_rate=16.7%, period=2025-03-03 → 2026-02-27
  macd_crossover: win_rate=21.1%, period=2024-03-01 → 2026-02-27
  rsi_crossover: win_rate=24.9%, period=2024-03-01 → 2026-02-27
  sma_crossover: win_rate=15.3%, period=2024-03-01 → 2026-02-27


In [16]:
# Load 1 file cụ thể
profile = SignalProfile.load(profiler.profile_dir / "sma_crossover.json")
profile.print_summary()

  SIGNAL PROFILE: sma_crossover
  Params:         {'period': 20, 'sl_pct': 0.07, 'tp_pct': 0.1}
  Symbols:        AAA, AAM, AAT, ABR, ABS, ABT, ACB, ACC, ACG, ACL, ADG, ADP, ADS, AFX, AGG, AGR, ANT, ANV, APG, APH, ASG, ASM, ASP, AST, BAF, BCE, BCG, BCM, BFC, BHN, BIC, BID, BKG, BMC, BMI, BMP, BRC, BSI, BSR, BTP, BTT, BVH, BWE, C32, C47, CCC, CCI, CCL, CDC, CHP, CIG, CII, CKG, CLC, CLL, CLW, CMG, CMV, CMX, CNG, COM, CRC, CRE, CRV, CSM, CSV, CTD, CTF, CTG, CTI, CTR, CTS, CVT, D2D, DAH, DAT, DBC, DBD, DBT, DC4, DCL, DCM, DGC, DGW, DHA, DHC, DHG, DHM, DIG, DLG, DMC, DPG, DPM, DPR, DQC, DRC, DRH, DRL, DSC, DSE, DSN, DTA, DTL, DTT, DVP, DXG, DXS, DXV, EIB, ELC, EVE, EVF, EVG, FCM, FCN, FDC, FIR, FIT, FMC, FPT, FRT, FTS, GAS, GDT, GEE, GEG, GEL, GEX, GHC, GIL, GMD, GMH, GSP, GTA, GVR, HAG, HAH, HAP, HAR, HAS, HAX, HCD, HCM, HDB, HDC, HDG, HHP, HHS, HHV, HID, HII, HMC, HNA, HPA, HPG, HPX, HQC, HRC, HSG, HSL, HT1, HTG, HTI, HTL, HTN, HTV, HU1, HUB, HVH, HVN, ICT, IDI, IJC, ILB, IMP, ITC, ITD, J

## 7. So sánh hiệu quả các strategy

Sử dụng kết quả profiling để so sánh nhanh.

In [17]:
# Bảng so sánh nhanh
comparison = []
for name, p in profiles.items():
    comparison.append({
        "Strategy": name,
        "Signals": p.total_signals,
        "Win Rate": f"{p.overall_win_rate:.1%}",
        "BUY Win%": f"{p.buy_stats.win_rate:.1%}",
        "Avg Return": f"{p.buy_stats.avg_return_pct:.2f}%",
        "R:R": f"{p.buy_stats.risk_reward:.2f}",
        "Frequency": f"{p.buy_stats.frequency:.4f}",
    })

import pandas as pd
pd.DataFrame(comparison).set_index("Strategy")

,Signals,Win Rate,BUY Win%,Avg Return,R:R,Frequency
Strategy,,,,,,
macd_crossover,11421,21.1%,32.4%,-0.28%,1.86,16.3980
rsi_crossover,2488,24.9%,44.9%,0.35%,1.34,4.8424
sma_crossover,17170,15.3%,27.2%,-0.28%,2.34,24.8465


## 8. Sử dụng PDFProfileReport trực tiếp

Ngoài `Profiler`, bạn có thể tạo PDF thủ công từ bất kỳ `BacktestReport` nào.

In [13]:
from vnstock_forecast.forecast.visualization.pdf_report import PDFProfileReport

# Lấy kết quả chi tiết của 1 technique từ profiler
result = profiler.results.get("sma_crossover")
if result:
    pdf = PDFProfileReport(
        technique_name=result.technique.name,
        technique_params=result.technique.params,
        description="SMA Crossover – giá cắt SMA tạo tín hiệu mua/bán",
        backtest_report=result.report,
        signal_profile=result.profile,
        signals=result.signals,
    )
    path = pdf.generate("outputs/sma_crossover_report.pdf")
    print(f"PDF created: {path}")

PDF created: outputs/sma_crossover_report.pdf


## Tổng kết

| Bước | API | Kết quả |
|------|-----|---------|
| Đăng ký strategy | `@register("tên")` trên class kế thừa `BaseTechnique` | Tự động vào registry |
| Xem registry | `list_technique_names()`, `get_all_techniques()` | Danh sách tên / dict class |
| Chạy profiling | `Profiler().run(data, techniques=[...])` | `dict[str, SignalProfile]` |
| Profile 1 strat | `profiler.run_single(technique, data)` | `SignalProfile` |
| Lưu JSON + PDF | `profiler.save()` | File `.json` + `.pdf` trong `profile/` |
| Lưu PDF riêng | `profiler.save_pdf("tên")` | File `.pdf` |
| Load profile | `SignalProfile.load(path)` / `.load_all(dir)` | `SignalProfile` instance |